In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 96.7 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import faiss

In [4]:
PROJECT_DIR = "/content/drive/MyDrive/PrismHire"

In [20]:
master_df = pd.read_csv(
    f"{PROJECT_DIR}/master_candidates.csv"
)

master_df.head()

,candidate_id,years_experience,current_title,current_industry,current_company_size,country,num_skills,avg_skill_proficiency,avg_skill_endorsements,avg_skill_duration,...,github_activity_score,connection_count,endorsements_received,notice_period_days,willing_to_relocate,preferred_work_mode,verified_email,verified_phone,linkedin_connected,avg_assessment_score
0,CAND_0000001,6.9,Backend Engineer,IT Services,10001+,Canada,17,2.235294,17.176471,24.823529,...,9.2,356,35,60,0,3,1,1,0,49.725
1,CAND_0000002,12.5,Operations Manager,IT Services,10001+,India,9,1.666667,7.777778,20.333333,...,0.0,179,3,60,0,4,0,0,0,0.000
2,CAND_0000003,1.1,Customer Support,IT Services,10001+,USA,6,1.500000,7.833333,17.666667,...,0.0,19,46,150,0,2,1,0,0,0.000
3,CAND_0000004,3.8,Marketing Manager,Paper Products,201-500,Australia,10,1.500000,8.000000,13.400000,...,0.0,485,22,120,1,3,1,1,1,0.000
4,CAND_0000005,11.0,Accountant,Manufacturing,1001-5000,India,6,1.666667,14.666667,24.500000,...,0.0,300,14,30,1,2,0,1,1,0.000


In [5]:
embeddings = np.load(
    f"{PROJECT_DIR}/models/candidate_embeddings.npy"
)

print(embeddings.shape)

(100000, 384)


In [21]:
print(master_df.shape)

(100000, 35)


In [6]:
embeddings = embeddings.astype("float32")

In [8]:
dimension = embeddings.shape[1]

print(dimension)

384


In [9]:
index = faiss.IndexFlatIP(
    dimension
)

In [10]:
index.add(
    embeddings
)

In [11]:
index.ntotal

100000

In [12]:
faiss.write_index(
    index,
    f"{PROJECT_DIR}/models/faiss_index.bin"
)

In [13]:
!pip install sentence-transformers -q

In [14]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
job_description = """
Senior Backend Engineer

Requirements:
5+ years experience
Python
REST APIs
Microservices
AWS
Docker
"""

In [17]:
job_embedding = model.encode(
    [job_description],
    normalize_embeddings=True
)

job_embedding = job_embedding.astype(
    "float32"
)

print(job_embedding.shape)

(1, 384)


In [18]:
scores, indices = index.search(
    job_embedding,
    k=10
)

print(scores)
print(indices)

[[0.6411643  0.6281549  0.6267394  0.6244748  0.6220093  0.62017775
  0.6200128  0.6186394  0.6178443  0.61777645]]
[[21574 68631 50623 51499 30215 15825 88423  1398 60362 71958]]


In [22]:
top_candidates = master_df.iloc[
    indices[0]
]

top_candidates[
    [
        "candidate_id",
        "current_title",
        "years_experience",
        "current_industry"
    ]
]

,candidate_id,current_title,years_experience,current_industry
21574,CAND_0021575,Cloud Engineer,7.1,Fintech
68631,CAND_0068632,Cloud Engineer,7.6,IT Services
50623,CAND_0050624,Cloud Engineer,4.0,E-commerce
51499,CAND_0051500,Cloud Engineer,5.6,IT Services
30215,CAND_0030216,Cloud Engineer,6.9,IT Services
15825,CAND_0015826,Cloud Engineer,3.8,IT Services
88423,CAND_0088424,Cloud Engineer,5.4,Food Delivery
1398,CAND_0001399,Cloud Engineer,3.8,E-commerce
60362,CAND_0060363,Cloud Engineer,1.9,IT Services
71958,CAND_0071959,Cloud Engineer,4.6,Food Delivery
